# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.02 · Cascada HF–Qwen en Colab

Ejecuta sin costo de API una primera pasada Qwen3-1.7B y dirige daño, dudas y controles a Qwen3-4B; es una alternativa experimental independiente de DeepSeek.

Qwen3 es una familia multilingüe de pesos abiertos [1]. Las revisiones exactas de `Qwen/Qwen3-1.7B` y `Qwen/Qwen3-4B` se fijan mediante sus tarjetas oficiales [2] [3]. La cascada carga los modelos secuencialmente en una NVIDIA L4: 1.7B aporta cobertura y 4B revisa daño, abstenciones, baja confianza y un control seguro. Esta política de enrutamiento es una decisión local que debe compararse con una pasada 4B y con referencias humanas; las propuestas no son *ground truth* [4].

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab` y asigne una **NVIDIA L4**. El notebook permanece local; Drive transporta solo versiones inmutables del bundle. Si la copia activa no coincide, la celda lee `bundle_releases/latest.json`, verifica todos sus SHA-256 y promueve automáticamente esa versión. Ejecute antes `02_00` directamente en Colab. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [5]. La integridad del bundle se comprueba con SHA-256 [6]. No sincronice cachés de modelos ni escriba checkpoints directamente en Drive.

In [ ]:
# Backend reproducible: local o Google Colab desde VS Code
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile

COLAB_NOTEBOOK_ID = "02_02"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = True
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "41d92ba615c4a4e5f72425376591ab7c88cb8707e003b350da0460af1141b073"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "eaa521a58c77e769c33829b96f9eb7f42cec38cfb79b6c3d95a5b36f2345c4c0"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if not manifest_path.is_file():
        return False
    try:
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            return False
        if manifest["bundle_id"] != expected_bundle_id:
            return False
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            return False
        return all(
            (bundle_dir / name).is_file() and _sha256(bundle_dir / name) == expected_sha256
            for name, expected_sha256 in _bundle_specs(manifest)
        )
    except (KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    latest_pointer = _read_manifest(latest_pointer_path) if latest_pointer_path.is_file() else {}
    latest_bundle_id = str(latest_pointer.get("bundle_id") or "")
    latest_matches_notebook = (
        len(latest_bundle_id) == 64
        and latest_bundle_id == COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        and latest_pointer.get("core_sha256") == COLAB_EXPECTED_CORE_SHA256
    )
    if latest_matches_notebook:
        release_source = "latest_pointer"
        RELEASE_DIR = RELEASES_DIR / latest_bundle_id
        expected_manifest_sha256 = latest_pointer.get("manifest_sha256")
    else:
        # Un cuaderno reproducible puede activar su release inmutable exacto aunque
        # latest todavía apunte a otra versión; jamás mezcla código e inputs.
        release_source = "notebook_pinned_release"
        latest_bundle_id = COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        RELEASE_DIR = RELEASES_DIR / latest_bundle_id
        pinned_manifest_path = RELEASE_DIR / "bundle_manifest.json"
        if not _bundle_is_current(RELEASE_DIR, pinned_manifest_path, latest_bundle_id):
            raise RuntimeError(
                "Drive no contiene ni latest compatible ni el release inmutable fijado por este "
                "cuaderno. Ejecute 02_00_preparacion_bundle_colab.ipynb y publique el bundle exacto."
            )
        pinned_manifest = _read_manifest(pinned_manifest_path)
        if pinned_manifest.get("core", {}).get("sha256") != COLAB_EXPECTED_CORE_SHA256:
            raise RuntimeError("El release fijado por el cuaderno contiene un core inesperado")
        expected_manifest_sha256 = _sha256(pinned_manifest_path)
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    if not release_manifest_path.is_file() or _sha256(release_manifest_path) != expected_manifest_sha256:
        raise RuntimeError("El manifiesto del release de Drive falta o no coincide con su referencia")
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive. Ejecute en Colab "
                "02_00_preparacion_bundle_colab.ipynb, confirme status=published_to_drive y "
                f"compruebe que exista {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'origen_del_release': release_source,
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Prompt operacional vigente', {'ruta': OPERATIONAL_PROMPT, 'versión': '3.2.0'}, tone='success')


## Configuración Colab y contrato JSON

In [ ]:
from moderacion_peru.providers import HuggingFaceProvider
from moderacion_peru.io import read_jsonl,write_json_atomic,write_jsonl_atomic

SOURCE=COLAB_CONTEXT.input('chunks_v2') if COLAB_CONTEXT else ROOT/'datos/processed/chunks_v2.jsonl'
CAMPAIGN_ROOT=COLAB_CONTEXT.scratch_output_dir if COLAB_CONTEXT else ROOT/'datos/etiquetado/cascada_qwen_hf'
CAMPAIGN_ROOT.mkdir(parents=True,exist_ok=True)
PRIMARY=CAMPAIGN_ROOT/'qwen3_1_7b_primary_v3_2.jsonl'
REVIEW=CAMPAIGN_ROOT/'qwen3_4b_review_v3_2.jsonl'
QUEUE=CAMPAIGN_ROOT/'qwen3_4b_review_queue_v3_2.jsonl'
RUN_PRIMARY=False
RUN_REVIEW=False
PRIMARY_LIMIT=20  # Smoke reanudable; use None para todos los pendientes.
REVIEW_LIMIT=20   # Smoke reanudable; use None para toda la cola dirigida.
REVIEW_CONFIDENCE_THRESHOLD=0.85
SAFE_CONTROL_RATE=0.05
MAX_NEEDS_REVIEW_FOR_QWEN4B=None

primary_provider=HuggingFaceProvider(model='Qwen/Qwen3-1.7B',revision='70d244cc86ccca08cf5af4e1e306ecf908b1ad5e',device='auto',records_per_request=5,inference_batch_size=4,max_new_tokens=256,operational_prompt_path=OPERATIONAL_PROMPT,label_source='qwen_hf_colab_primary')
reviewer_provider=HuggingFaceProvider(model='Qwen/Qwen3-4B',revision='1cfa9a7208912126459214e8b04321603b3df60c',device='auto',records_per_request=5,inference_batch_size=2,max_new_tokens=256,operational_prompt_path=OPERATIONAL_PROMPT,label_source='qwen_hf_colab_review')
primary_probe=primary_provider.probe(); reviewer_probe=reviewer_provider.probe()
for role,probe in [('primario',primary_probe),('revisor',reviewer_probe)]:
    if probe['response_format']!={'type':'json_object'} or probe['output_contract']['root_key']!='annotations':
        raise RuntimeError(f'El proveedor {role} no garantiza el wrapper JSON annotations')
    if Path(probe['operational_prompt_path']).resolve()!=OPERATIONAL_PROMPT.resolve():
        raise RuntimeError(f'El proveedor {role} no usa el prompt operacional vigente')
show_result('Qwen3-1.7B primario',primary_probe,tone='success')
show_result('Qwen3-4B revisor',reviewer_probe,tone='success')
show_callout('Optimización correcta','02_02 no llama a DeepSeek y no genera cargos ni aciertos de caché de su API. Agrupa 5 chunks por prompt y aprovecha la L4 con lotes GPU de 4 prompts para 1.7B y 2 para 4B; subirlos exige medir memoria antes.',tone='neutral')
show_callout('Contrato verificado','Ambos modelos reciben el prompt operacional 3.2.0 y solo se persisten objetos validados con raíz annotations, orden y chunk_id exactos.',tone='success')

## Primera pasada Qwen3-1.7B

In [ ]:
from tqdm.auto import tqdm
from moderacion_peru.labeling import annotate_batched_incremental

progress={'bar':None,'description':''}
def qwen_progress(description):
    progress['description']=description
    def callback(event):
        if event['status']=='started':
            progress['bar']=tqdm(total=event['selected'],desc=description,unit='chunk'); return
        bar=progress.get('bar')
        if bar is not None and event.get('advance'):
            bar.update(event['advance']); bar.set_postfix(ok=event['labeled'],errores=event['errors'])
        if event['status'] in {'finished','interrupted_checkpoint'} and bar is not None:
            bar.close(); progress['bar']=None
    return callback

if RUN_PRIMARY:
    try:
        primary_result=annotate_batched_incremental(read_jsonl(SOURCE),primary_provider,PRIMARY,error_path=PRIMARY.with_suffix('.errors.jsonl'),limit=PRIMARY_LIMIT,processing_batch_size=20,progress_callback=qwen_progress('Qwen3-1.7B'),run_metadata={'provider':primary_probe,'role':'qwen_primary','prompt_version':'3.2.0'})
    finally:
        primary_provider.unload()
    show_result('Primera pasada persistida',primary_result,tone='success')
else:
    primary_provider.unload()
    show_callout('Primera pasada desactivada','Use 20 para el smoke y después None; la salida reanuda por chunk_id.',tone='neutral')

## Enrutamiento reproducible hacia Qwen3-4B

In [ ]:
from moderacion_peru.labeling_calibration import build_directed_review_queue

if PRIMARY.is_file():
    primary_rows=list(read_jsonl(PRIMARY))
    primary_ids={row['chunk_id'] for row in primary_rows}
    paired_chunks=[row for row in read_jsonl(SOURCE) if row['chunk_id'] in primary_ids]
    review_queue,routing=build_directed_review_queue(paired_chunks,primary_rows,confidence_threshold=REVIEW_CONFIDENCE_THRESHOLD,safe_control_rate=SAFE_CONTROL_RATE,max_needs_review=MAX_NEEDS_REVIEW_FOR_QWEN4B,seed=42)
    write_jsonl_atomic(QUEUE,review_queue)
    write_json_atomic(CAMPAIGN_ROOT/'qwen_routing_summary_v3_2.json',routing)
    show_result('Cola Qwen3-4B',routing,tone='success')
else:
    review_queue=[]
    show_callout('Falta primera pasada','Ejecute Qwen3-1.7B antes de construir la cola 4B.',tone='warning')

## Revisión dirigida Qwen3-4B y checkpoint

In [ ]:
if RUN_REVIEW:
    if not QUEUE.is_file(): raise FileNotFoundError('Falta la cola dirigida Qwen3-4B')
    try:
        review_result=annotate_batched_incremental(read_jsonl(QUEUE),reviewer_provider,REVIEW,error_path=REVIEW.with_suffix('.errors.jsonl'),limit=REVIEW_LIMIT,processing_batch_size=10,progress_callback=qwen_progress('Qwen3-4B'),run_metadata={'provider':reviewer_probe,'role':'qwen_directed_review','prompt_version':'3.2.0'})
    finally:
        reviewer_provider.unload()
    show_result('Revisión 4B persistida',review_result,tone='success')
    if COLAB_CONTEXT is not None:
        from moderacion_peru.colab import publish_colab_outputs
        show_result('Checkpoint publicado en Drive',publish_colab_outputs(COLAB_CONTEXT),tone='success')
else:
    reviewer_provider.unload()
    show_callout('Revisión 4B desactivada','La cola incluye daño, abstenciones, confianza menor que 0.85 y un control seguro reproducible del 5%.',tone='neutral')

## Publicación o checkpoint en Drive

Los archivos se generan en el SSD efímero de `/content`. Active esta celda después de un checkpoint coherente o al finalizar; publica un solo TAR.GZ y luego su manifiesto.

In [ ]:
PUBLISH_TO_DRIVE = False
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None and globals().get('AUTO_PUBLISH_CHECKPOINTS'):
    show_callout('Checkpoint automático activo', 'La recuperación, los checkpoints periódicos, Ctrl+C y cada cierre de campaña ya publican un TAR.GZ atómico en Drive.', tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación desactivada', 'Cambie PUBLISH_TO_DRIVE=True tras guardar un checkpoint consistente.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] A. Yang, A. Li, B. Yang, et al., "Qwen3 Technical Report," arXiv:2505.09388, 2025, doi: 10.48550/arXiv.2505.09388.

[2] Qwen Team, "Model Card: Qwen/Qwen3-1.7B," Hugging Face Hub, revision 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-1.7B/tree/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e. Accessed: Aug. 7, 2026.

[3] Qwen Team, "Model Card: Qwen/Qwen3-4B," Hugging Face Hub, revision 1cfa9a7208912126459214e8b04321603b3df60c, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-4B/tree/1cfa9a7208912126459214e8b04321603b3df60c. Accessed: Aug. 5, 2026.

[4] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.

[5] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[6] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.